In [108]:
import pandas as pd 

In [ ]:
def create_clean_df_for_Content_pipeline(movies_df,tags_df):
    new_movies_df = movies_df[['movieId','title','genres']].copy()
    # cleaning movies_df
    new_movies_df['title'] = (
        movies_df['title'].fillna("")
        .astype(str)
        .str.strip()
    )
    new_movies_df['genres'] = (
        new_movies_df['genres'].fillna("")
        .astype(str)
        .str.replace("|"," ")
        .str.strip()
    )
    
    # cleaning tags_df
    new_tags_df = tags_df[["movieId","tag"]].copy()
    new_tags_df.dropna(subset=['movieId','tag'])
    
    new_tags_df['tag'] = (
    new_tags_df['tag']
    .astype(str)
    .str.strip()
    .str.lower()
    )
    
    new_tags_df = new_tags_df[new_tags_df['tag'] != ""]
    
    tags_grouped = (
    new_tags_df.groupby('movieId')['tags']
    .apply(lambda crr_mov_tags : " ".join(crr_mov_tags)) # this lambda function input tags of first movie and join them with " " and then moves to next movie tags and do same upto so on
    .reset_index(name='tags') # it cut out the movieId from the resultant series into separate column and the name="tags" changes the name of remaining tag  series to tagss
)
    
    # merging the tags and movies_df
    movies_df_final = pd.merge(
        new_movies_df,
        tags_grouped,
        on='movieId',
        how='left',
        validate='one_to_one' # validating so that each movieId appears one in both data frames otherwise it will raise error 
    )
    
    # cleaning final frame 
    movies_df_final["tags"] = movies_df_final['tags'].fillna("")
    movies_df_final = movies_df_final.drop_duplicates(subset=["movieId"])
    movies_df_final = movies_df_final.reset_index(drop=True)
    
    return movies_df_final


In [121]:
movies = pd.read_csv(r'../datasets/movies.csv')
tags = pd.read_csv(r"../datasets/tags.csv")

In [122]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 9722 entries, 0 to 9721
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9722 non-null   int64
 1   title    9722 non-null   str  
 2   genres   9722 non-null   str  
 3   year     9722 non-null   int64
 4   imdbId   9722 non-null   int64
 5   tmdbId   9722 non-null   int64
dtypes: int64(4), str(2)
memory usage: 455.8 KB


In [123]:
tags

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200
...,...,...,...,...
3678,606,7382,for katie,1171234019
3679,606,7936,austere,1173392334
3680,610,3265,gun fu,1493843984
3681,610,3265,heroic bloodshed,1493843978


In [ ]:
tags_grouped = (
    tags.groupby('movieId')['tags']
    .apply(lambda crr_mov_tags : " ".join(crr_mov_tags)) # this lambda function input tags of first movie and join them with " " and then moves to next movie tags and do same upto so on
    .reset_index(name='tags') # it cut out the movieId from the resultant series into separate column and the name="tags" changes the name of remaining tag  series to tagss
)

In [112]:
tmp = create_clean_df_for_Content_pipeline(movies,tags)

In [113]:
tmp

,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9717,193581,Black Butler: Book of the Atlantic (2017),Action Animation Comedy Fantasy
9718,193583,No Game No Life: Zero (2017),Animation Comedy Fantasy
9719,193585,Flint (2017),Drama
9720,193587,Bungo Stray Dogs: Dead Apple (2018),Action Animation
